# Part 1: ADQL queries through Rust+DataFusion

In [1]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
from hats_query import ParquetReader
from tqdm.asyncio import tqdm

In [3]:
parquet_reader = ParquetReader(
    "/Users/hombit/Downloads/gaia_dr3-Norder=2-Npix=99-page128k.parquet",
    sql="SELECT ra, dec, pm FROM gaia",
    table_name="gaia",
)
async for batch in tqdm(parquet_reader):
    pass
batch

126it [00:00, 298.13it/s]


pyarrow.RecordBatch
ra: double
dec: double
pm: float
----
ra: [178.61961877649642,178.6069303420108,178.64660648956783,178.64306865786418,178.6204774373651,178.6421684676145,178.68918891690603,178.70020488547058,178.69376521806717,178.68398670253185,...,180.02015052410053,180.03297364626522,180.00966847037105,180.00625420822766,179.9863362521926,179.97363581007824,179.9627249096502,180.0112053206381,179.99521907174724,179.9982823938733]
dec: [-1.1982950766478953,-1.1942681935573485,-1.183847015082675,-1.180190415490132,-1.1752141313397662,-1.16859653803089,-1.1782542376723306,-1.1747244190140036,-1.1747006804259013,-1.1672096185876224,...,-0.0409608538677676,-0.0365275661563278,-0.0376915039946868,-0.0356005824083088,-0.0446413655785876,-0.0425017464541571,-0.034451704875626,-0.0130381569933629,-0.0148023438350006,-0.002115523265182]
pm: [14.477711,7.5275464,9.449141,null,null,6.8775434,48.33449,5.834092,2.7215068,11.552052,...,null,null,23.508802,11.557439,5.459875,7.464184,4.240536,4

In [4]:
parquet_reader = ParquetReader(
    "/Users/hombit/Downloads/gaia_dr3-Norder=2-Npix=99-page128k.parquet",
    sql="SELECT TOP 5 source_id, RAND() as random_number FROM gaia OFFSET 20",
    table_name="gaia",
)
async for batch in tqdm(parquet_reader):
    pass
batch

1it [00:00, 26.44it/s]


pyarrow.RecordBatch
source_id: int64
random_number: double
----
source_id: [3566851145396284416,3566851149691341184,3566851179755353472,3566851179755760128,3566851184050649216]
random_number: [0.2986898427753202,0.6245737613687059,0.16879886971158942,0.6349369048754919,0.051148243766433565]

# Part 2: Web server

In [5]:
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from IPython.display import HTML, Markdown

In [6]:
Markdown(f"```toml\n{Path('hats-query.example.toml').read_text()}\n```")

```toml
#   python -m hats_query --config hats-query.toml

[server]
address = "0.0.0.0"
port = 80
# Reload on code changes and return tracebacks in error responses
developer_mode = false
# Who runs this deployment
contact = "ops@example.org"

[log]
# critical, error, warning, info, debug, or trace
level = "info"

# Directories this service reads, maps "source" URL to "path" endpoint.
# Relative sources are resolved against this file's directory: these are the example
# catalogs from the hats git submodule, fetch it with `git submodule update --init`.
[[mount]]
path = "/hats"
source = "tests/data/git-submodules/hats/tests/data"

# HATS catalogs published over TAP
[[tap.table]]
name = "small_sky.object"
# A collection, with an index on "id"
path = "/hats/small_sky_o1_collection"

[[tap.table]]
name = "small_sky.source"
path = "/hats/small_sky_source"

# Queries offered at /tap/examples, "tables" optionally lists the tables each is about
[[tap.example]]
name = "Objects in a cone"
tables = ["small_sky.object"]
query = """
SELECT id, ra, dec
FROM small_sky.object
WHERE 1 = CONTAINS(POINT(ra, dec), CIRCLE(315.0, -45.0, 5.0))
"""

[[tap.example]]
name = "Object by id"
tables = ["small_sky.object"]
query = """
SELECT id, ra, dec
FROM small_sky.object
WHERE id = 707
"""

```

In [9]:
server = subprocess.Popen([sys.executable, "-m", "hats_query", "--config", "hats-query.example.toml", "--server.port", "8765"])
time.sleep(1)

INFO:     Started server process [91796]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8765 (Press CTRL+C to quit)


In [10]:
HTML(urllib.request.urlopen("http://127.0.0.1:8765/tap/examples").read().decode())

INFO:     127.0.0.1:53076 - "GET /tap/examples HTTP/1.1" 200 OK


In [11]:
server.terminate()



INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [91796]
